# Préparation des données pour la modélisation

Cette étape sépare les données d'entraînement et de test avant les transformations. Le `ColumnTransformer` sera intégré aux pipelines des modèles afin d'éviter toute fuite de données.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].replace(r"^\s*$", pd.NA, regex=True),
    errors="coerce",
)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print(df.shape)
print(df["Churn"].value_counts(dropna=False))

(7043, 20)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [2]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Taille train :", X_train.shape)
print("Taille test :", X_test.shape)
print("Proportion churn train :", round(y_train.mean(), 4))
print("Proportion churn test :", round(y_test.mean(), 4))

Taille train : (5634, 19)
Taille test : (1409, 19)
Proportion churn train : 0.2654
Proportion churn test : 0.2654


In [3]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]
categorical_features = X.select_dtypes(include="object").columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

print("Variables numériques :", numeric_features)
print("Nombre de variables catégorielles :", len(categorical_features))
print("Préprocesseur prêt :", type(preprocessor).__name__)

Variables numériques : ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Nombre de variables catégorielles : 15
Préprocesseur prêt : ColumnTransformer


C:\Users\Hp\AppData\Local\Temp\ipykernel_6540\1918351870.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include="object").columns.tolist()


## Régression logistique

La régression logistique constitue un modèle de référence interprétable pour une classification binaire. `class_weight="balanced"` augmente le poids de la classe churn afin de ne pas privilégier artificiellement la classe majoritaire.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
        ),
    ]
)

logistic_pipeline.fit(X_train, y_train)
y_pred = logistic_pipeline.predict(X_test)
y_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))
print("\nMétriques sur le jeu de test :")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Précision churn : {precision_score(y_test, y_pred):.3f}")
print(f"Rappel churn : {recall_score(y_test, y_pred):.3f}")
print(f"F1-score churn : {f1_score(y_test, y_pred):.3f}")
print(f"AUC : {roc_auc_score(y_test, y_proba):.3f}")
print("\nRapport de classification :")
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))

Matrice de confusion :
[[747 288]
 [ 81 293]]

Métriques sur le jeu de test :
Accuracy : 0.738
Précision churn : 0.504
Rappel churn : 0.783
F1-score churn : 0.614
AUC : 0.841

Rapport de classification :
              precision    recall  f1-score   support

    No churn       0.90      0.72      0.80      1035
       Churn       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



## Ridge Classifier

Le Ridge Classifier fournit un second modèle linéaire de référence. La régularisation L2 limite les coefficients trop instables après l'encodage des variables catégorielles.

In [11]:
from sklearn.linear_model import RidgeClassifier

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RidgeClassifier(class_weight="balanced", random_state=42),
        ),
    ]
)

ridge_pipeline.fit(X_train, y_train)
ridge_pred = ridge_pipeline.predict(X_test)
ridge_scores = ridge_pipeline.decision_function(X_test)

print("Matrice de confusion :")
print(confusion_matrix(y_test, ridge_pred))
print("\nMétriques sur le jeu de test :")
print(f"Accuracy : {accuracy_score(y_test, ridge_pred):.3f}")
print(f"Précision churn : {precision_score(y_test, ridge_pred):.3f}")
print(f"Rappel churn : {recall_score(y_test, ridge_pred):.3f}")
print(f"F1-score churn : {f1_score(y_test, ridge_pred):.3f}")
print(f"AUC : {roc_auc_score(y_test, ridge_scores):.3f}")

Matrice de confusion :
[[744 291]
 [ 79 295]]

Métriques sur le jeu de test :
Accuracy : 0.737
Précision churn : 0.503
Rappel churn : 0.789
F1-score churn : 0.615
AUC : 0.836


## Arbre de décision

L'arbre de décision permet de représenter des règles de décision. Une profondeur maximale de 5 limite sa complexité pour réduire le surapprentissage.

In [5]:
from sklearn.tree import DecisionTreeClassifier

tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                class_weight="balanced",
                max_depth=5,
                random_state=42,
            ),
        ),
    ]
)

tree_pipeline.fit(X_train, y_train)
tree_pred = tree_pipeline.predict(X_test)
tree_proba = tree_pipeline.predict_proba(X_test)[:, 1]

print("Matrice de confusion :")
print(confusion_matrix(y_test, tree_pred))
print("\nMétriques sur le jeu de test :")
print(f"Accuracy : {accuracy_score(y_test, tree_pred):.3f}")
print(f"Précision churn : {precision_score(y_test, tree_pred):.3f}")
print(f"Rappel churn : {recall_score(y_test, tree_pred):.3f}")
print(f"F1-score churn : {f1_score(y_test, tree_pred):.3f}")
print(f"AUC : {roc_auc_score(y_test, tree_proba):.3f}")

Matrice de confusion :
[[780 255]
 [ 90 284]]

Métriques sur le jeu de test :
Accuracy : 0.755
Précision churn : 0.527
Rappel churn : 0.759
F1-score churn : 0.622
AUC : 0.832


In [10]:
from sklearn.tree import export_text

tree_classifier = tree_pipeline.named_steps["classifier"]
tree_preprocessor = tree_pipeline.named_steps["preprocessor"]
tree_feature_names = tree_preprocessor.get_feature_names_out()
tree_rules = export_text(
    tree_classifier,
    feature_names=list(tree_feature_names),
    max_depth=3,
)

print("Profondeur réelle de l'arbre :", tree_classifier.get_depth())
print("Nombre de feuilles :", tree_classifier.get_n_leaves())
print("Règles principales (trois premiers niveaux) :")
print(tree_rules)

train_tree_score = tree_pipeline.score(X_train, y_train)
test_tree_score = tree_pipeline.score(X_test, y_test)
print(f"Accuracy train : {train_tree_score:.3f}")
print(f"Accuracy test : {test_tree_score:.3f}")

Profondeur réelle de l'arbre : 5
Nombre de feuilles : 31
Règles principales (trois premiers niveaux) :
|--- categorical__Contract_Month-to-month <= 0.50
|   |--- numeric__MonthlyCharges <= 0.95
|   |   |--- categorical__OnlineSecurity_No <= 0.50
|   |   |   |--- categorical__Contract_Two year <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- categorical__Contract_Two year >  0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |--- categorical__OnlineSecurity_No >  0.50
|   |   |   |--- categorical__Contract_Two year <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- categorical__Contract_Two year >  0.50
|   |   |   |   |--- truncated branch of depth 2
|   |--- numeric__MonthlyCharges >  0.95
|   |   |--- categorical__Contract_Two year <= 0.50
|   |   |   |--- categorical__StreamingMovies_No <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- categorical__StreamingMovies_No >  0.50
|   |   |   |   |--- trunc

## Random Forest

Le Random Forest combine plusieurs arbres entraînés sur des échantillons différents. Cette combinaison réduit généralement la variance d'un arbre seul et peut améliorer la généralisation.

In [6]:
from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_pipeline.fit(X_train, y_train)
forest_pred = random_forest_pipeline.predict(X_test)
forest_proba = random_forest_pipeline.predict_proba(X_test)[:, 1]

print("Matrice de confusion :")
print(confusion_matrix(y_test, forest_pred))
print("\nMétriques sur le jeu de test :")
print(f"Accuracy : {accuracy_score(y_test, forest_pred):.3f}")
print(f"Précision churn : {precision_score(y_test, forest_pred):.3f}")
print(f"Rappel churn : {recall_score(y_test, forest_pred):.3f}")
print(f"F1-score churn : {f1_score(y_test, forest_pred):.3f}")
print(f"AUC : {roc_auc_score(y_test, forest_proba):.3f}")

Matrice de confusion :
[[768 267]
 [ 78 296]]

Métriques sur le jeu de test :
Accuracy : 0.755
Précision churn : 0.526
Rappel churn : 0.791
F1-score churn : 0.632
AUC : 0.842


In [12]:
comparison = pd.DataFrame(
    {
        "Modèle": [
            "Régression logistique",
            "Ridge Classifier",
            "Arbre de décision",
            "Random Forest",
        ],
        "Accuracy": [
            accuracy_score(y_test, y_pred),
            accuracy_score(y_test, ridge_pred),
            accuracy_score(y_test, tree_pred),
            accuracy_score(y_test, forest_pred),
        ],
        "Précision churn": [
            precision_score(y_test, y_pred),
            precision_score(y_test, ridge_pred),
            precision_score(y_test, tree_pred),
            precision_score(y_test, forest_pred),
        ],
        "Rappel churn": [
            recall_score(y_test, y_pred),
            recall_score(y_test, ridge_pred),
            recall_score(y_test, tree_pred),
            recall_score(y_test, forest_pred),
        ],
        "F1-score churn": [
            f1_score(y_test, y_pred),
            f1_score(y_test, ridge_pred),
            f1_score(y_test, tree_pred),
            f1_score(y_test, forest_pred),
        ],
        "AUC": [
            roc_auc_score(y_test, y_proba),
            roc_auc_score(y_test, ridge_scores),
            roc_auc_score(y_test, tree_proba),
            roc_auc_score(y_test, forest_proba),
        ],
    }
)

print(comparison.set_index("Modèle").round(3))

                       Accuracy  Précision churn  Rappel churn  \
Modèle                                                           
Régression logistique     0.738            0.504         0.783   
Ridge Classifier          0.737            0.503         0.789   
Arbre de décision         0.755            0.527         0.759   
Random Forest             0.755            0.526         0.791   

                       F1-score churn    AUC  
Modèle                                        
Régression logistique           0.614  0.841  
Ridge Classifier                0.615  0.836  
Arbre de décision               0.622  0.832  
Random Forest                   0.632  0.842  


## Validation croisée et recherche sur grille

La recherche utilise uniquement `X_train` et `y_train`. Le rappel churn est optimisé avec une validation croisée stratifiée à 5 plis. Le jeu de test reste réservé à l'évaluation finale.

In [9]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [6, 10],
    "classifier__min_samples_leaf": [1, 3],
}

grid_search = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=param_grid,
    scoring="recall",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)

best_forest = grid_search.best_estimator_
best_pred = best_forest.predict(X_test)
best_proba = best_forest.predict_proba(X_test)[:, 1]

print("Meilleurs paramètres :")
print(grid_search.best_params_)
print(f"Meilleur rappel moyen en validation croisée : {grid_search.best_score_:.3f}")
print("\nPerformances finales sur le jeu de test :")
print(f"Accuracy : {accuracy_score(y_test, best_pred):.3f}")
print(f"Précision churn : {precision_score(y_test, best_pred):.3f}")
print(f"Rappel churn : {recall_score(y_test, best_pred):.3f}")
print(f"F1-score churn : {f1_score(y_test, best_pred):.3f}")
print(f"AUC : {roc_auc_score(y_test, best_proba):.3f}")

Meilleurs paramètres :
{'classifier__max_depth': 6, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 200}
Meilleur rappel moyen en validation croisée : 0.809

Performances finales sur le jeu de test :
Accuracy : 0.746
Précision churn : 0.514
Rappel churn : 0.794
F1-score churn : 0.624
AUC : 0.841
